<a href="https://colab.research.google.com/github/mw12anek/Computer-Science-class/blob/main/Dogs_and_cats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CATS VS. DOGS IMAGE CLASSIFICATION WITH TENSORFLOW / KERAS
# ============================================================

import os
import urllib.request
import zipfile
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models


# ============================================================
# 1. SETTINGS
# ============================================================

URL = (
    "https://download.microsoft.com/download/3/E/1/"
    "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/"
    "kagglecatsanddogs_5340.zip"
)

ZIP_FILE = "cats_and_dogs.zip"
EXTRACT_DIR = "cats_and_dogs_data"

IMG_SIZE = (200, 200)
BATCH_SIZE = 32
SEED = 1337
EPOCHS = 10


# ============================================================
# 2. DOWNLOAD DATASET
# ============================================================

if not os.path.exists(ZIP_FILE):
    print("Downloading Cats vs. Dogs dataset...")
    urllib.request.urlretrieve(URL, ZIP_FILE)
    print("Download complete.")
else:
    print("ZIP file already exists. Skipping download.")


# ============================================================
# 3. EXTRACT DATASET
# ============================================================

if not os.path.exists(EXTRACT_DIR):
    print("Extracting dataset...")

    with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

    print("Extraction complete.")
else:
    print("Dataset already extracted.")


# ============================================================
# 4. FIND THE PetImages DIRECTORY
# ============================================================

# Microsoft ZIP normally extracts as:
#
# cats_and_dogs_data/
#     PetImages/
#         Cat/
#         Dog/

BASE_DIR = os.path.join(EXTRACT_DIR, "PetImages")

if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"Could not find dataset directory: {BASE_DIR}"
    )

print("\nDataset directory:", BASE_DIR)


# ============================================================
# 5. REMOVE CORRUPTED / INVALID IMAGES
# ============================================================

print("\nChecking images for corrupted files...")

valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
removed_files = 0

for class_name in ["Cat", "Dog"]:

    class_dir = os.path.join(BASE_DIR, class_name)

    for file_name in os.listdir(class_dir):

        file_path = os.path.join(class_dir, file_name)

        if not os.path.isfile(file_path):
            continue

        extension = Path(file_path).suffix.lower()

        # Remove unsupported file types
        if extension not in valid_extensions:
            os.remove(file_path)
            removed_files += 1
            continue

        # Try decoding the image using TensorFlow
        try:
            image_data = tf.io.read_file(file_path)

            image = tf.io.decode_image(
                image_data,
                channels=3,
                expand_animations=False
            )

            # Force TensorFlow to actually evaluate the image
            _ = image.shape

        except Exception:
            print("Removing invalid image:", file_path)
            os.remove(file_path)
            removed_files += 1


print(f"\nRemoved {removed_files} invalid/corrupted files.")


# ============================================================
# 6. CREATE TRAINING DATASET
# ============================================================

print("\nCreating training dataset...")

train_dataset = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)


# ============================================================
# 7. CREATE VALIDATION DATASET
# ============================================================

print("\nCreating validation dataset...")

val_dataset = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)


# ============================================================
# 8. DISPLAY CLASS NAMES
# ============================================================

class_names = train_dataset.class_names

print("\nClasses:", class_names)


# ============================================================
# 9. IMPROVE DATA PIPELINE PERFORMANCE
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

val_dataset = val_dataset.prefetch(
    buffer_size=AUTOTUNE
)


# ============================================================
# 10. BUILD CONVOLUTIONAL NEURAL NETWORK
# ============================================================

model = models.Sequential([

    # Explicit input layer
    layers.Input(shape=(200, 200, 3)),

    # Convert pixel values from 0-255 to 0-1
    layers.Rescaling(1.0 / 255),

    # --------------------------------------------------------
    # First Convolutional Block
    # --------------------------------------------------------

    layers.Conv2D(
        32,
        kernel_size=(3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D(
        pool_size=(2, 2)
    ),

    # --------------------------------------------------------
    # Second Convolutional Block
    # --------------------------------------------------------

    layers.Conv2D(
        64,
        kernel_size=(3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D(
        pool_size=(2, 2)
    ),

    # --------------------------------------------------------
    # Third Convolutional Block
    # --------------------------------------------------------

    layers.Conv2D(
        128,
        kernel_size=(3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D(
        pool_size=(2, 2)
    ),

    # --------------------------------------------------------
    # Convert feature maps into a vector
    # --------------------------------------------------------

    layers.Flatten(),

    # --------------------------------------------------------
    # Fully Connected Classification Layer
    # --------------------------------------------------------

    layers.Dense(
        128,
        activation="relu"
    ),

    # Reduce overfitting
    layers.Dropout(0.5),

    # --------------------------------------------------------
    # Binary Output
    # --------------------------------------------------------

    layers.Dense(
        1,
        activation="sigmoid"
    )
])


# ============================================================
# 11. DISPLAY MODEL ARCHITECTURE
# ============================================================

print("\nMODEL ARCHITECTURE")
print("=" * 60)

model.summary()


# ============================================================
# 12. COMPILE MODEL
# ============================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(),

    loss=tf.keras.losses.BinaryCrossentropy(),

    metrics=["accuracy"]
)


# ============================================================
# 13. TRAIN MODEL
# ============================================================

print("\nStarting training...\n")

history = model.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=EPOCHS
)


# ============================================================
# 14. FINAL VALIDATION RESULTS
# ============================================================

val_loss, val_accuracy = model.evaluate(
    val_dataset,
    verbose=0
)

print("\n" + "=" * 60)
print("FINAL VALIDATION RESULTS")
print("=" * 60)

print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(
    f"Validation Accuracy: "
    f"{val_accuracy * 100:.2f}%"
)


# ============================================================
# 15. DISPLAY TRAINING HISTORY
# ============================================================

print("\nTRAINING HISTORY")

for epoch in range(EPOCHS):

    train_acc = history.history["accuracy"][epoch]
    val_acc = history.history["val_accuracy"][epoch]

    train_loss = history.history["loss"][epoch]
    val_loss_epoch = history.history["val_loss"][epoch]

    print(
        f"Epoch {epoch + 1:2d}: "
        f"Train Accuracy = {train_acc:.4f}, "
        f"Validation Accuracy = {val_acc:.4f}, "
        f"Train Loss = {train_loss:.4f}, "
        f"Validation Loss = {val_loss_epoch:.4f}"
    )

Download complete.
Extracting dataset...
Extraction complete.

Dataset directory: cats_and_dogs_data/PetImages

Checking images for corrupted files...
Removing invalid image: cats_and_dogs_data/PetImages/Cat/10404.jpg
Removing invalid image: cats_and_dogs_data/PetImages/Cat/4351.jpg
Removing invalid image: cats_and_dogs_data/PetImages/Cat/666.jpg
Removing invalid image: cats_and_dogs_data/PetImages/Dog/2317.jpg
Removing invalid image: cats_and_dogs_data/PetImages/Dog/11702.jpg
Removing invalid image: cats_and_dogs_data/PetImages/Dog/11233.jpg
Removing invalid image: cats_and_dogs_data/PetImages/Dog/11912.jpg
Removing invalid image: cats_and_dogs_data/PetImages/Dog/9500.jpg
Removing invalid image: cats_and_dogs_data/PetImages/Dog/2494.jpg

Removed 11 invalid/corrupted files.

Creating training dataset...
Found 24991 files belonging to 2 classes.
Using 19993 files for training.

Creating validation dataset...
Found 24991 files belonging to 2 classes.
Using 4998 files for validation.

Cla

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 200, 200, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 198, 198, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 99, 99, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 97, 97, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 48, 48, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 46, 46, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 23, 23, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 67712)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     8,667,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,760,641 (33.42 MB)

 Trainable params: 8,760,641 (33.42 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 61ms/step - accuracy: 0.6276 - loss: 0.6405 - val_accuracy: 0.7197 - val_loss: 0.5777
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 69s 50ms/step - accuracy: 0.7318 - loss: 0.5334 - val_accuracy: 0.7809 - val_loss: 0.4714
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 33s 53ms/step - accuracy: 0.7857 - loss: 0.4548 - val_accuracy: 0.7715 - val_loss: 0.4825
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.8267 - loss: 0.3850 - val_accuracy: 0.8069 - val_loss: 0.4297
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 60ms/step - accuracy: 0.8631 - loss: 0.3147 - val_accuracy: 0.8071 - val_loss: 0.4663
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 33s 53ms/step - accuracy: 0.8970 - loss: 0.2489 - val_accuracy: 0.8123 - val_loss: 0.4991
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 54ms/step - accuracy: 0.9255 - loss: 0.1879 - val_accuracy: 0.8189 - val_loss: 0.5909
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0